In [ ]:
# 1. 표준 라이브러리
import os
import json
import random

# 2. 서드파티 라이브러리 (데이터 처리 및 시각화)
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import pandas as pd

# 3. PyTorch 관련 라이브러리
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR

# 4. 기타 외부 라이브러리 (timm, ssim)
import timm
!pip install pytorch-msssim  # 필요 시 주석 해제 후 실행
from pytorch_msssim import ssim
from google.colab import files
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# 모델설정

In [ ]:
# git clone 으로 깃허브 가져오기
!git clone https://github.com/JingyunLiang/SwinIR.git
# 해당 폴더로 경로바꾸기
%cd SwinIR

In [ ]:
# 모델아키텍처 불러오기
from models.network_swinir import SwinIR

"""
  window_size - 한 attention 범위 (M x M 윈도우)
  embed_dim   - feature 차원 크기
  depths      - 각 RSTB 블록당 STL 레이어 수, len(depths) = RSTB 블록 수
  num_heads   - 각 STL의 attention head 수
  upscale     - 업스케일 배율 (복원 태스크는 1)
  upsampler   - 업샘플링 방식 (복원 태스크는 '')
  img_size    - 학습 패치 크기 (crop_size와 동일하게)
"""

SWINIR_CONFIG = dict(
    upscale=2,
    in_chans=3,
    img_size=64,
    window_size=8,
    img_range=1.,
    depths=[6, 6, 6, 6],
    embed_dim=60,
    num_heads=[6, 6, 6, 6],
    mlp_ratio=2,
    upsampler='pixelshuffledirect',
    resi_connection='1conv'
)
%cd /content

# 열화전처리

In [ ]:
def degrade_image_x2(hr):
    h, w = hr.shape[:2]

    # HR 256x256 → LR 128x128
    lr = cv2.resize(
        hr,
        (w // 2, h // 2),
        interpolation=cv2.INTER_AREA
    )

    # 약한 blur
    if random.random() < 0.20:
        k = random.choice([3, 5])
        lr = cv2.GaussianBlur(
            lr,
            (k, k),
            sigmaX=random.uniform(0.6, 1.4)
        )

    # 약한 JPEG
    if random.random() < 0.25:
        quality = random.randint(50, 85)

        lr_bgr = cv2.cvtColor(lr, cv2.COLOR_RGB2BGR)
        success, encoded = cv2.imencode(
            ".jpg",
            lr_bgr,
            [int(cv2.IMWRITE_JPEG_QUALITY), quality]
        )

        if success:
            lr = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
            lr = cv2.cvtColor(lr, cv2.COLOR_BGR2RGB)

    # 약한 noise
    if random.random() < 0.10:
        lr_f = lr.astype(np.float32) / 255.0
        sigma = random.uniform(0.003, 0.015)
        noise = np.random.normal(0, sigma, lr_f.shape).astype(np.float32)
        lr = np.clip(lr_f + noise, 0.0, 1.0)
        lr = (lr * 255.0).astype(np.uint8)

    return lr

In [ ]:
class SROIEDataset(Dataset):
    def __init__(self, root_dir, file_list, crop_size=256, min_text_ratio=0.005, max_crop_tries=10):
        self.root_dir = root_dir
        self.file_list = file_list
        self.crop_size = crop_size
        self.min_text_ratio = min_text_ratio
        self.max_crop_tries = max_crop_tries

        self.img_dir = os.path.join(root_dir, "img")
        self.box_dir = os.path.join(root_dir, "box")
        self.ent_dir = os.path.join(root_dir, "entities")

    def __len__(self):
        return len(self.file_list)

    def _build_mask_from_box(self, box_path, h, w):
        mask = np.zeros((h, w), dtype=np.float32)

        lines = None
        for encoding in ["utf-8", "cp949", "latin-1"]:
            try:
                with open(box_path, "r", encoding=encoding) as f:
                    lines = f.readlines()
                break
            except UnicodeDecodeError:
                continue

        if lines is None:
            return mask

        for line in lines:
            parts = line.rstrip("\n").split(",")
            if len(parts) < 8:
                continue

            try:
                coords = list(map(int, parts[:8]))
            except ValueError:
                continue

            pts = np.array([
                [coords[0], coords[1]],
                [coords[2], coords[3]],
                [coords[4], coords[5]],
                [coords[6], coords[7]]
            ], dtype=np.int32)

            cv2.fillPoly(mask, [pts], 1.0)

        return mask

    def _random_crop(self, img, mask):
        crop_size = self.crop_size
        h, w = img.shape[:2]

        if h < crop_size or w < crop_size:
            new_h = max(h, crop_size)
            new_w = max(w, crop_size)

            img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
            mask = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

            h, w = img.shape[:2]

        best_img_crop = None
        best_mask_crop = None
        best_ratio = -1.0

        for _ in range(self.max_crop_tries):
            top = random.randint(0, h - crop_size) if h > crop_size else 0
            left = random.randint(0, w - crop_size) if w > crop_size else 0

            img_crop = img[top:top + crop_size, left:left + crop_size]
            mask_crop = mask[top:top + crop_size, left:left + crop_size]

            text_ratio = float(mask_crop.mean())

            if text_ratio > best_ratio:
                best_img_crop = img_crop
                best_mask_crop = mask_crop
                best_ratio = text_ratio

            if text_ratio >= self.min_text_ratio:
                return img_crop, mask_crop

        return best_img_crop, best_mask_crop

    def __getitem__(self, idx):
        sample_id = self.file_list[idx]

        img_path = os.path.join(self.img_dir, sample_id + ".jpg")
        box_path = os.path.join(self.box_dir, sample_id + ".txt")

        hr = cv2.imread(img_path)
        hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)

        h, w = hr.shape[:2]
        mask = self._build_mask_from_box(box_path, h, w)

        hr, mask = self._random_crop(hr, mask)

        lr = degrade_image_x2(hr)

        hr = torch.from_numpy(
            hr.astype(np.float32) / 255.0
        ).permute(2, 0, 1).contiguous()

        lr = torch.from_numpy(
            lr.astype(np.float32) / 255.0
        ).permute(2, 0, 1).contiguous()

        mask = torch.from_numpy(
            mask.astype(np.float32)
        ).unsqueeze(0).contiguous()

        return lr, hr, mask, sample_id

In [ ]:
# =============================
# 1. Kaggle 데이터셋 다운로드
# =============================
from google.colab import files
import os
from sklearn.model_selection import train_test_split

files.upload()  # kaggle.json 업로드

os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d urbikn/sroie-datasetv2
!unzip -q sroie-datasetv2.zip


# =============================
# 2. 경로 설정
# =============================
train_root_dir = "SROIE2019/train"
test_root_dir  = "SROIE2019/test"

train_img_dir = os.path.join(train_root_dir, "img")
train_box_dir = os.path.join(train_root_dir, "box")
train_ent_dir = os.path.join(train_root_dir, "entities")

test_img_dir = os.path.join(test_root_dir, "img")
test_box_dir = os.path.join(test_root_dir, "box")


# =============================
# 3. train 파일 목록 생성
# img + box + entities 모두 있는 샘플만 사용
# =============================
img_files = {
    os.path.splitext(f)[0]
    for f in os.listdir(train_img_dir)
    if f.lower().endswith(".jpg")
}

box_files = {
    os.path.splitext(f)[0]
    for f in os.listdir(train_box_dir)
    if f.lower().endswith(".txt")
}

ent_files = {
    os.path.splitext(f)[0]
    for f in os.listdir(train_ent_dir)
    if f.lower().endswith(".txt")
}

common_files = sorted(img_files & box_files & ent_files)

print("Common train samples:", len(common_files))


# =============================
# 4. train / val 파일명 기준 분리
# =============================
train_files, val_files = train_test_split(
    common_files,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_files = sorted(train_files)
val_files = sorted(val_files)

print("Train size:", len(train_files))
print("Val size:", len(val_files))


# =============================
# 5. test 파일 목록 생성
# test는 img + box가 모두 있는 샘플만 사용
# entities는 test에 없을 수 있으므로 제외
# =============================
test_img_files = {
    os.path.splitext(f)[0]
    for f in os.listdir(test_img_dir)
    if f.lower().endswith(".jpg")
}

test_box_files = {
    os.path.splitext(f)[0]
    for f in os.listdir(test_box_dir)
    if f.lower().endswith(".txt")
}

test_files = sorted(test_img_files & test_box_files)

print("Test size:", len(test_files))

In [ ]:
# =============================
# 6. Dataset / Loader 생성
# =============================

BATCH_SIZE = 32  # OOM 나면 4로 낮추기

train_dataset = SROIEDataset(
    root_dir=train_root_dir,
    file_list=train_files,
    crop_size=192,
    min_text_ratio=0.005,
    max_crop_tries=10
)
val_dataset = SROIEDataset(
    root_dir=train_root_dir,
    file_list=val_files,
    crop_size=192
)

test_dataset = SROIEDataset(
    root_dir=test_root_dir,
    file_list=test_files,
    crop_size=192
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=False
)

In [ ]:
# 마스크 영역에서만 L1 loss 계산
def masked_l1(pred, target, mask, eps=1e-8):
    diff = torch.abs(pred - target)
    masked_diff = diff * mask
    denom = mask.sum() * pred.size(1) + eps
    return masked_diff.sum() / denom


# 마스크 영역에서만 MSE 계산
def masked_mse(pred, target, mask, eps=1e-8):
    diff2 = (pred - target) ** 2
    masked_diff2 = diff2 * mask
    denom = mask.sum() * pred.size(1) + eps
    return masked_diff2.sum() / denom


# MSE → PSNR 변환
def psnr_from_mse(mse, max_val=1.0):
    return 10.0 * torch.log10((max_val ** 2) / (mse + 1e-8))


# 텍스트 영역 bbox crop 기반 SSIM loss
def masked_bbox_ssim_loss(pred, target, mask, pad=4):
    """
    pred, target: (B, C, H, W)
    mask: (B, 1, H, W) 또는 (B, C, H, W)

    기존 pred * mask 방식은 검은 배경이 많이 들어가므로,
    마스크가 존재하는 영역만 bbox로 잘라 SSIM을 계산함.
    """
    losses = []
    B, C, H, W = pred.shape

    if mask.size(1) != 1:
        mask_1ch = mask[:, :1]
    else:
        mask_1ch = mask

    for b_idx in range(B):
        pos = torch.where(mask_1ch[b_idx, 0] > 0)

        if pos[0].numel() == 0:
            continue

        y_min = max(0, int(pos[0].min()) - pad)
        y_max = min(H, int(pos[0].max()) + pad + 1)
        x_min = max(0, int(pos[1].min()) - pad)
        x_max = min(W, int(pos[1].max()) + pad + 1)

        pred_crop = pred[b_idx:b_idx+1, :, y_min:y_max, x_min:x_max]
        target_crop = target[b_idx:b_idx+1, :, y_min:y_max, x_min:x_max]

        # pytorch_msssim 기본 window_size=11보다 작으면 SSIM 계산 불안정
        if pred_crop.shape[-1] < 11 or pred_crop.shape[-2] < 11:
            continue

        ssim_val = ssim(
            pred_crop,
            target_crop,
            data_range=1.0,
            size_average=True
        )

        losses.append(1.0 - ssim_val)

    if len(losses) == 0:
        return pred.new_tensor(0.0)

    return torch.stack(losses).mean()


# 학습에 사용할 loss들을 모두 계산
def compute_losses(pred, target, mask, a=2.0, b=0.05, c=0.1, d=0.02):
    """
    a: 텍스트 영역 L1 가중치
    b: 배경 영역 L1 가중치
    c: 텍스트 영역 SSIM loss 가중치
    d: 전체 이미지 SSIM loss 가중치
    """
    bg_mask = 1.0 - mask

    # L1
    l1_global = F.l1_loss(pred, target)
    l1_text = masked_l1(pred, target, mask)
    l1_bg = masked_l1(pred, target, bg_mask)

    # global SSIM
    ssim_global = ssim(pred, target, data_range=1.0, size_average=True)
    ssim_global_loss = 1.0 - ssim_global

    # text SSIM: mask 곱 방식이 아니라 bbox crop 기반
    ssim_text_loss = masked_bbox_ssim_loss(pred, target, mask)
    ssim_text = 1.0 - ssim_text_loss

    # loss 후보들
    loss_1 = l1_global
    loss_2 = l1_global + d * ssim_global_loss
    loss_3 = a * l1_text + b * l1_bg + d * ssim_global_loss
    loss_4 = a * l1_text + b * l1_bg + c * ssim_text_loss + d * ssim_global_loss

    return {
        "l1_global": l1_global,
        "l1_text": l1_text,
        "l1_bg": l1_bg,
        "ssim_global": ssim_global,
        "ssim_text": ssim_text,
        "ssim_text_loss": ssim_text_loss,
        "loss_1": loss_1,
        "loss_2": loss_2,
        "loss_3": loss_3,
        "loss_4": loss_4,
    }


# 평가 지표 계산
def compute_metrics(pred, target, mask):
    mse_global = F.mse_loss(pred, target)
    psnr_global = psnr_from_mse(mse_global)
    ssim_global = ssim(pred, target, data_range=1.0, size_average=True)

    mse_text = masked_mse(pred, target, mask)
    psnr_text = psnr_from_mse(mse_text)

    # 평가용 text SSIM도 bbox crop 기반으로 계산
    ssim_text_loss = masked_bbox_ssim_loss(pred, target, mask)
    ssim_text = 1.0 - ssim_text_loss

    return {
        "psnr_global": psnr_global.item(),
        "ssim_global": ssim_global.item(),
        "psnr_text": psnr_text.item(),
        "ssim_text": ssim_text.item(),
    }

In [ ]:
!mkdir -p /content/pretrained

!wget -O /content/pretrained/002_lightweightSR_DIV2K_s64w8_SwinIR-S_x2.pth \
https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/002_lightweightSR_DIV2K_s64w8_SwinIR-S_x2.pth


# Optuna 하이퍼파라미터 탐색

pretrained SwinIR-S x2 fine-tuning의 val **ssim_text**를 최대화하는 loss 가중치(b, c, d)와 learning rate를 Optuna로 자동 탐색한다.

Adam은 loss 전체 스케일에 거의 불변이라 (a, b, c, d)는 비율만 의미가 있으므로 a=2.0으로 고정하고 나머지 4차원(b, c, d, lr)만 탐색한다.

In [ ]:
!pip install -q optuna

import random
import numpy as np
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner


def train_for_optuna(lr, b, c, d, max_epochs=15, patience=4, trial=None, save_path=None, save_path_psnr=None):
    # -------------------
    # seed 고정 (조합 간 공정 비교)
    # -------------------
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    # -------------------
    # Model 생성 + pretrained weight load
    # -------------------
    model = SwinIR(**SWINIR_CONFIG).to(device)

    pretrained_path = "/content/pretrained/002_lightweightSR_DIV2K_s64w8_SwinIR-S_x2.pth"
    state = torch.load(pretrained_path, map_location=device)

    if "params_ema" in state:
        state = state["params_ema"]
    elif "params" in state:
        state = state["params"]

    model.load_state_dict(state, strict=True)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min=1e-6)

    best_ssim_text = -float("inf")
    best_psnr_text = -float("inf")
    best_epoch = -1
    best_psnr_epoch = -1
    no_improve = 0

    history = {
        "val_ssim_text": [],
        "val_psnr_text": [],
    }

    trial_tag = f"trial {trial.number}" if trial is not None else "final"

    for epoch in range(max_epochs):

        # -------------------
        # Train
        # -------------------
        model.train()

        for lr_img, hr, mask, _ in tqdm(train_loader, desc=f"[{trial_tag}] Train Epoch {epoch+1}"):

            lr_img = lr_img.to(device)
            hr = hr.to(device)
            mask = mask.to(device)

            pred = model(lr_img)

            losses = compute_losses(pred, hr, mask, a=2.0, b=b, c=c, d=d)
            loss = losses["loss_4"]

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        scheduler.step()

        # -------------------
        # Validation
        # -------------------
        model.eval()

        val_sums = {"psnr_text": 0.0, "ssim_text": 0.0}
        n_text = 0

        with torch.no_grad():

            for lr_img, hr, mask, _ in tqdm(val_loader, desc=f"[{trial_tag}] Val Epoch {epoch+1}"):

                lr_img = lr_img.to(device)
                hr = hr.to(device)
                mask = mask.to(device)

                pred = model(lr_img)

                batch_size = lr_img.size(0)

                for i in range(batch_size):
                    metrics = compute_metrics(pred[i:i+1], hr[i:i+1], mask[i:i+1])

                    val_sums["psnr_text"] += metrics["psnr_text"]
                    val_sums["ssim_text"] += metrics["ssim_text"]
                    n_text += 1

        val_ssim_text = val_sums["ssim_text"] / n_text
        val_psnr_text = val_sums["psnr_text"] / n_text

        history["val_ssim_text"].append(val_ssim_text)
        history["val_psnr_text"].append(val_psnr_text)

        # -------------------
        # Best 갱신 체크 + 얼리스탑 counter
        # -------------------
        if val_ssim_text > best_ssim_text:
            best_ssim_text = val_ssim_text
            best_epoch = epoch + 1
            no_improve = 0

            if save_path is not None:
                torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1

        # psnr_text 기준 best는 별도 추적/저장 (얼리스탑에는 관여 X)
        if val_psnr_text > best_psnr_text:
            best_psnr_text = val_psnr_text
            best_psnr_epoch = epoch + 1

            if save_path_psnr is not None:
                torch.save(model.state_dict(), save_path_psnr)

        print(
            f"[{trial_tag}] Epoch {epoch+1}/{max_epochs} | "
            f"val ssim_text: {val_ssim_text:.4f} | "
            f"val psnr_text: {val_psnr_text:.2f} | "
            f"best: {best_ssim_text:.4f} (epoch {best_epoch})"
        )

        # -------------------
        # Optuna pruning
        # -------------------
        if trial is not None:
            trial.report(val_ssim_text, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

        # -------------------
        # 얼리스탑
        # -------------------
        if no_improve >= patience:
            break

    return {
        "best_ssim_text": best_ssim_text,
        "best_psnr_text": best_psnr_text,
        "best_epoch": best_epoch,
        "best_psnr_epoch": best_psnr_epoch,
        "history": history,
    }

In [ ]:
def objective(trial):
    lr = trial.suggest_float("lr", 5e-6, 2e-4, log=True)
    b = trial.suggest_float("b", 0.005, 0.5, log=True)
    c = trial.suggest_float("c", 0.01, 1.0, log=True)
    d = trial.suggest_float("d", 0.005, 0.3, log=True)

    result = train_for_optuna(lr, b, c, d, max_epochs=15, patience=4, trial=trial)
    return result["best_ssim_text"]


sampler = TPESampler(seed=42)
pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=3)

study = optuna.create_study(
    study_name="swinir_ssim_text",
    storage="sqlite:///optuna_swinir.db",
    load_if_exists=True,
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
)

study.optimize(objective, n_trials=30)

print("Best value (val ssim_text):", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
best = study.best_params

result = train_for_optuna(
    **best,
    max_epochs=100,
    patience=10,
    save_path="best_swinir_optuna.pt",
    save_path_psnr="best_swinir_optuna_psnr.pt",
)

print("최종 학습 완료")
print("best val ssim_text:", result["best_ssim_text"], f"(epoch {result['best_epoch']})")
print("best val psnr_text:", result["best_psnr_text"], f"(epoch {result['best_psnr_epoch']})")
print("저장: best_swinir_optuna.pt (ssim 기준) / best_swinir_optuna_psnr.pt (psnr 기준)")

# 평가

In [ ]:
def evaluate_metrics(model, data_loader, desc="Test", is_srcnn=False):
    model.eval()

    sums = {
        "psnr_global": 0.0,
        "ssim_global": 0.0,
        "psnr_text": 0.0,
        "ssim_text": 0.0,
    }

    n = 0

    with torch.no_grad():
        for lr_img, hr, mask, _ in tqdm(data_loader, desc=desc):
            lr_img = lr_img.to(device)
            hr = hr.to(device)
            mask = mask.to(device)

            if is_srcnn:
                lr_up = F.interpolate(lr_img, size=hr.shape[-2:], mode="bicubic", align_corners=False)
                pred = model(lr_up)
            else:
                pred = model(lr_img)

            pred = torch.clamp(pred, 0.0, 1.0)

            batch_size = lr_img.size(0)

            for i in range(batch_size):
                metrics = compute_metrics(
                    pred[i:i+1],
                    hr[i:i+1],
                    mask[i:i+1]
                )

                sums["psnr_global"] += metrics["psnr_global"]
                sums["ssim_global"] += metrics["ssim_global"]
                sums["psnr_text"] += metrics["psnr_text"]
                sums["ssim_text"] += metrics["ssim_text"]

                n += 1

    return {
        "psnr_global": sums["psnr_global"] / n,
        "ssim_global": sums["ssim_global"] / n,
        "psnr_text": sums["psnr_text"] / n,
        "ssim_text": sums["ssim_text"] / n,
    }

# =============================
# 평가 실행
# =============================

# --- Optuna best SwinIR ---
for ckpt_type, path in [
    ("SSIM-best", "best_swinir_optuna.pt"),
    ("PSNR-best", "best_swinir_optuna_psnr.pt"),
]:
    model = SwinIR(**SWINIR_CONFIG).to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    result = evaluate_metrics(model, test_loader, desc=f"SwinIR Optuna {ckpt_type}")
    print(f"\n[SwinIR Optuna {ckpt_type}]")
    print(f"  PSNR global: {result['psnr_global']:.4f}")
    print(f"  SSIM global: {result['ssim_global']:.4f}")
    print(f"  PSNR text  : {result['psnr_text']:.4f}")
    print(f"  SSIM text  : {result['ssim_text']:.4f}")


# OCR

In [ ]:
# ============================================================
# OCR Evaluation Code
# - EasyOCR 고정
# - 전체 영수증 단위 OCR 평가
# - Metrics: CER, WER
# - Baselines: LR_raw, LR_Bicubic, HR_Original
# ============================================================

# =============================
# 0. 설치 및 import
# =============================
!pip -q install easyocr pandas tqdm

import os
import re
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from tqdm import tqdm
import easyocr


# =============================
# 1. OCR Reader 생성
# =============================
reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())


# =============================
# 2. 텍스트 정규화 함수
# =============================
def normalize_text(text):
    text = text.upper()
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text


# =============================
# 3. GT 텍스트 읽기
# =============================
def read_gt_text_from_box(box_path):
    items = []

    lines = None
    for enc in ["utf-8", "cp949", "latin-1"]:
        try:
            with open(box_path, "r", encoding=enc) as f:
                lines = f.readlines()
            break
        except UnicodeDecodeError:
            continue

    if lines is None:
        return ""

    for line in lines:
        line = line.rstrip("\n")
        parts = line.split(",", 8)

        if len(parts) < 9:
            continue

        try:
            coords = list(map(int, parts[:8]))
        except ValueError:
            continue

        text = parts[8].strip()
        if text == "":
            continue

        xs = coords[0::2]
        ys = coords[1::2]

        x_center = sum(xs) / 4
        y_center = sum(ys) / 4

        items.append((y_center, x_center, text))

    items = sorted(items, key=lambda x: (x[0], x[1]))

    gt_text = " ".join([item[2] for item in items])
    return normalize_text(gt_text)


# =============================
# 4. OCR 실행 함수
# =============================
def run_easyocr(img_np):
    if img_np.dtype != np.uint8:
        img = np.clip(img_np, 0, 1)
        img = (img * 255).astype(np.uint8)
    else:
        img = img_np.copy()

    results = reader.readtext(
        img,
        detail=1,
        paragraph=False
    )

    ocr_items = []

    for bbox, text, conf in results:
        if text is None or text.strip() == "":
            continue

        xs = [p[0] for p in bbox]
        ys = [p[1] for p in bbox]

        x_center = sum(xs) / 4
        y_center = sum(ys) / 4

        ocr_items.append((y_center, x_center, text))

    ocr_items = sorted(ocr_items, key=lambda x: (x[0], x[1]))

    pred_text = " ".join([item[2] for item in ocr_items])
    return normalize_text(pred_text)


# =============================
# 5. Edit Distance
# =============================
def edit_distance(seq1, seq2):
    n = len(seq1)
    m = len(seq2)

    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n + 1):
        dp[i][0] = i

    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if seq1[i - 1] == seq2[j - 1] else 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )

    return dp[n][m]


# =============================
# 6. CER / WER / OCR Accuracy
# =============================
def compute_ocr_metrics(pred_text, gt_text):
    pred_text = normalize_text(pred_text)
    gt_text = normalize_text(gt_text)

    gt_chars = list(gt_text)
    pred_chars = list(pred_text)

    char_dist = edit_distance(pred_chars, gt_chars)
    cer = char_dist / max(len(gt_chars), 1)

    gt_words = gt_text.split()
    pred_words = pred_text.split()

    word_dist = edit_distance(pred_words, gt_words)
    wer = word_dist / max(len(gt_words), 1)

    return {
        "CER": cer,
        "WER": wer
    }


# =============================
# 7. sliding window SR 함수
# =============================
def sliding_window_sr(model, lr_np, window_size=96, stride=96, is_srcnn=False):
    h, w = lr_np.shape[:2]
    new_h = ((h + window_size - 1) // window_size) * window_size
    new_w = ((w + window_size - 1) // window_size) * window_size

    lr_resized = cv2.resize(lr_np, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
    lr_t = torch.from_numpy(
        lr_resized.astype(np.float32) / 255.0
    ).permute(2, 0, 1).unsqueeze(0).to(device)

    out_h, out_w = new_h * 2, new_w * 2
    output = torch.zeros(1, 3, out_h, out_w).to(device)
    count  = torch.zeros(1, 1, out_h, out_w).to(device)

    # 패치 + 좌표 수집
    patches, coords = [], []
    for top in range(0, new_h, stride):
        for left in range(0, new_w, stride):
            top_end    = min(top + window_size, new_h)
            left_end   = min(left + window_size, new_w)
            top_start  = top_end - window_size
            left_start = left_end - window_size
            patch = lr_t[:, :, top_start:top_end, left_start:left_end].squeeze(0)
            if is_srcnn:
                patch = F.interpolate(
                    patch.unsqueeze(0),
                    size=(window_size*2, window_size*2),
                    mode="bicubic", align_corners=False
                ).squeeze(0)
            patches.append(patch)
            coords.append((top_start, left_start))

    # 배치 추론
    model.eval()
    with torch.no_grad():
        batch = torch.stack(patches).to(device)
        preds = torch.clamp(model(batch), 0.0, 1.0)
        for j, pred in enumerate(preds):
            ts, ls = coords[j]
            o_t, o_l = ts*2, ls*2
            output[:, :, o_t:o_t+pred.shape[1], o_l:o_l+pred.shape[2]] += pred.unsqueeze(0)
            count[:, :,  o_t:o_t+pred.shape[1], o_l:o_l+pred.shape[2]] += 1.0

    result = (output / count).squeeze(0).permute(1, 2, 0).cpu().numpy()
    result = cv2.resize(result, (w*2, h*2), interpolation=cv2.INTER_CUBIC)
    return np.clip(result, 0.0, 1.0)


# =============================
# 8. 모델 로드 함수
# =============================
def load_swinir_model(ckpt_path):
    model = SwinIR(**SWINIR_CONFIG).to(device)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=True)
    model.eval()
    return model


# =============================
# 9. 비교할 모델 목록
# =============================
MODEL_CONFIGS = [
    {
        "name": "SwinIR_optuna_SSIMbest",
        "type": "swinir",
        "ckpt": "best_swinir_optuna.pt"
    },
    {
        "name": "SwinIR_optuna_PSNRbest",
        "type": "swinir",
        "ckpt": "best_swinir_optuna_psnr.pt"
    },
]


# =============================
# 10. 한 이미지 평가 결과 추가 함수
# =============================
def add_ocr_result(all_rows, sample_id, method_name, img_np, gt_text, save_images=False, save_dir=None):
    pred_text = run_easyocr(img_np)
    metrics = compute_ocr_metrics(pred_text, gt_text)

    row = {
        "sample_id": sample_id,
        "method": method_name,
        "CER": metrics["CER"],
        "WER": metrics["WER"],
        "gt_text": gt_text,
        "pred_text": pred_text
    }

    all_rows.append(row)

    # 즉시 파일 저장
    pd.DataFrame([row]).to_csv(
        "/content/ocr_detail_results.csv",
        mode="a",
        header=not os.path.exists("/content/ocr_detail_results.csv"),
        index=False
    )
# =============================
# 11. OCR 평가 메인 함수
# =============================
def evaluate_ocr_on_testset(
    max_samples=None,
    save_images=False,
    save_dir="/content/ocr_eval_images"
):
    if save_images:
        os.makedirs(save_dir, exist_ok=True)

    # -----------------------------
    # 모델 미리 로드
    # -----------------------------
    loaded_models = []

    for cfg in MODEL_CONFIGS:
        ckpt_path = cfg["ckpt"]

        if not os.path.exists(ckpt_path):
            print(f"[SKIP] checkpoint 없음: {ckpt_path}")
            continue

        model = load_swinir_model(ckpt_path)

        loaded_models.append({
            "name": cfg["name"],
            "type": cfg["type"],
            "model": model
        })

        print(f"[LOAD] {cfg['name']} <- {ckpt_path}")

    print(f"\nLoaded models: {len(loaded_models)}")

    # -----------------------------
    # 평가 대상 샘플
    # -----------------------------
    eval_files = sampled_files

    all_rows = []

    # -----------------------------
    # 샘플별 OCR 평가
    # -----------------------------
    for sample_id in tqdm(eval_files, desc="OCR Evaluation"):

        img_path = os.path.join(test_root_dir, "img", sample_id + ".jpg")
        box_path = os.path.join(test_root_dir, "box", sample_id + ".txt")

        if not os.path.exists(img_path) or not os.path.exists(box_path):
            continue

        # HR 원본
        hr_np = cv2.imread(img_path)
        hr_np = cv2.cvtColor(hr_np, cv2.COLOR_BGR2RGB)

        h, w = hr_np.shape[:2]

        # GT 텍스트
        gt_text = read_gt_text_from_box(box_path)

        if gt_text == "":
            continue

        # LR 생성
        lr_np = degrade_image_x2(hr_np)

        # ------------------------------------
        # Baseline 1: LR raw
        # ------------------------------------
        add_ocr_result(
            all_rows=all_rows,
            sample_id=sample_id,
            method_name="LR_raw",
            img_np=lr_np,
            gt_text=gt_text,
            save_images=save_images,
            save_dir=save_dir
        )

        # ------------------------------------
        # Baseline 2: LR bicubic upsampled
        # ------------------------------------
        lr_bicubic = cv2.resize(
            lr_np,
            (w, h),
            interpolation=cv2.INTER_CUBIC
        )

        add_ocr_result(
            all_rows=all_rows,
            sample_id=sample_id,
            method_name="LR_Bicubic",
            img_np=lr_bicubic,
            gt_text=gt_text,
            save_images=save_images,
            save_dir=save_dir
        )

        # ------------------------------------
        # Baseline 3: HR original
        # ------------------------------------
        add_ocr_result(
            all_rows=all_rows,
            sample_id=sample_id,
            method_name="HR_Original",
            img_np=hr_np,
            gt_text=gt_text,
            save_images=save_images,
            save_dir=save_dir
        )

        # ------------------------------------
        # SR 모델들 평가
        # ------------------------------------
        for item in loaded_models:
            model_name = item["name"]
            model_type = item["type"]
            model = item["model"]

            is_srcnn = (model_type == "srcnn")

            sr_img = sliding_window_sr(
                model=model,
                lr_np=lr_np,
                window_size=96,
                stride=96,
                is_srcnn=is_srcnn
            )

            sr_uint8 = (np.clip(sr_img, 0, 1) * 255).astype(np.uint8)

            add_ocr_result(
                all_rows=all_rows,
                sample_id=sample_id,
                method_name=model_name,
                img_np=sr_uint8,
                gt_text=gt_text,
                save_images=save_images,
                save_dir=save_dir
            )

    # -----------------------------
    # DataFrame 변환
    # -----------------------------
    detail_df = pd.DataFrame(all_rows)

    if len(detail_df) == 0:
        print("평가 결과가 비어 있음. test_files, test_root_dir, box 파일을 확인해야 함.")
        return None, None

    summary_df = (
        detail_df
        .groupby("method")[["CER", "WER"]]
        .mean()
        .reset_index()
        .sort_values(by="CER", ascending=True)
    )

    return detail_df, summary_df

# =============================
# 12. 전체 테스트
# =============================
random.seed(42)
sampled_files = random.sample(test_files, min(100, len(test_files)))

detail_df, summary_df = evaluate_ocr_on_testset(
    max_samples=None,
    save_images=False
)

print("\n===== OCR Summary: Full Test =====")
display(summary_df)

detail_df.to_csv("/content/ocr_detail_results.csv", index=False)
summary_df.to_csv("/content/ocr_summary_results.csv", index=False)

print("Saved:")
print("/content/ocr_detail_results.csv")
print("/content/ocr_summary_results.csv")